<div style="font-size:2em; font-weight:bold; margin-bottom:8px;">04 — Enrich Chunks with Metadata</div>

This notebook reads the chunks produced by notebook 03 and attaches the **metadata** that downstream retrieval services use to filter, rank, debug, and cite results.

It is **Step 4** of the RAG data indexing pipeline — metadata enrichment only.

---

**What this notebook does:**
1. Loads the chunks from `data/processed/03_chunks.jsonl`
2. Explains the target metadata schema
3. Defines small helper functions to build a stable `chunk_id` and a metadata record
4. Tests the helpers on a single chunk
5. Enriches every chunk with metadata
6. Validates that every chunk has all required fields and a unique `chunk_id`
7. Saves the result to `data/processed/04_chunks_with_metadata.jsonl`
8. Reads the output back to verify it looks correct

**What this notebook intentionally does NOT do:**
- No embedding (that is notebook 05)
- No Qdrant indexing (that is notebook 06)

> **Before running:** make sure notebook 03 has produced `data/processed/03_chunks.jsonl`.

---
## 1. Imports

We need a small set of tools:
- **`json`** — read and write JSON Lines files
- **`hashlib`** — build a stable, repeatable `chunk_id` from the chunk content
- **`datetime`** — record when each chunk was indexed (`created_at`)
- **`pathlib.Path`** — OS-independent file paths

In [1]:
# ── [1 / 10] Imports ────────────────────────────────────────────────────────

import json
import hashlib
from datetime import datetime, timezone
from pathlib import Path

print("Imports ready.")

Imports ready.


---
## 2. Configuration — Input and Output Paths

All paths live in one place. The same project-root detection used in the earlier
notebooks is reused here.

In [2]:
# ── [2 / 10] Configuration ──────────────────────────────────────────────────

# A short label describing what kind of document these chunks come from
DOCUMENT_TYPE = "scientific_abstract"

# Resolve the project root no matter where the notebook is run from
_cwd = Path.cwd()
ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd

# Input — produced by notebook 03
INPUT_FILE = ROOT / "data" / "processed" / "03_chunks.jsonl"

# Output — enriched chunks for the embedding step
OUTPUT_DIR  = ROOT / "data" / "processed"
OUTPUT_FILE = OUTPUT_DIR / "04_chunks_with_metadata.jsonl"

print(f"Project root : {ROOT}")
print(f"Input file   : {INPUT_FILE}")
print(f"Output file  : {OUTPUT_FILE}")
print(f"Input exists : {INPUT_FILE.exists()}")

Project root : /app
Input file   : /app/data/processed/03_chunks.jsonl
Output file  : /app/data/processed/04_chunks_with_metadata.jsonl
Input exists : True


---
## 3. Load the Chunks

We read the JSONL file produced by notebook 03. Each chunk currently has:
`document_id`, `chunk_index`, `text`, `title`, `source`, `dataset_config`.

If the input file is missing, we stop with a clear error.

In [3]:
# ── [3 / 10] Load the Chunks ────────────────────────────────────────────────

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Input file not found: {INPUT_FILE}\n"
        "Run notebook 03_chunk_corpus.ipynb first."
    )

chunks = []
with INPUT_FILE.open("r", encoding="utf-8") as f:
    for line in f:
        chunks.append(json.loads(line))

print(f"Loaded {len(chunks):,} chunks")
print()
print("Example chunk keys:", list(chunks[0].keys()))

Loaded 12,281 chunks

Example chunk keys: ['document_id', 'chunk_index', 'text', 'title', 'source', 'dataset_config']


---
## 4. The Target Metadata Schema

Each enriched chunk will carry the metadata recommended in the project goal.
Good metadata is what lets a retrieval service filter, rank, debug, and cite chunks.

| Field | Source | Why it matters |
|---|---|---|
| `chunk_id` | hash of `document_id` + `chunk_index` + text | Unique, stable primary key for the vector store |
| `document_id` | from chunk | Groups chunks that came from the same document |
| `chunk_index` | from chunk | Order of the chunk inside its document |
| `source_file` | `source` field | Where the document originally came from |
| `document_type` | config | Lets retrieval filter by content type |
| `title` | from chunk | Human-readable context + citation |
| `created_at` | now (UTC) | When this chunk was indexed (debugging / freshness) |
| `text` | from chunk | The chunk content itself (what gets embedded) |

> We keep `chapter` / `section` out of this dataset because SciFact abstracts have
> no chapter structure. The helper still leaves room to add them later.

In [4]:
# ── [4 / 10] Show the Target Schema On Screen ───────────────────────────────

# Just print an example of the shape we are about to build, so the goal is clear.
schema_example = {
    "chunk_id"      : "<sha1 hex>",
    "document_id"   : "<from chunk>",
    "chunk_index"   : 0,
    "source_file"   : "<source>",
    "document_type" : DOCUMENT_TYPE,
    "title"         : "<from chunk>",
    "created_at"    : "<UTC ISO 8601 timestamp>",
    "text"          : "<chunk text>",
}

print("Target enriched-chunk schema:")
print(json.dumps(schema_example, indent=2))

Target enriched-chunk schema:
{
  "chunk_id": "<sha1 hex>",
  "document_id": "<from chunk>",
  "chunk_index": 0,
  "source_file": "<source>",
  "document_type": "scientific_abstract",
  "title": "<from chunk>",
  "created_at": "<UTC ISO 8601 timestamp>",
  "text": "<chunk text>"
}


---
## 5. Helper Functions — Stable IDs and Metadata

We define two small, single-purpose functions:

| Function | Responsibility |
|---|---|
| `make_chunk_id` | Build a stable SHA-1 id from `document_id`, `chunk_index`, and text |
| `build_metadata` | Turn a raw chunk into the full enriched record |

**Why a content-based id?** Hashing the content means re-running the pipeline on the
same data always produces the same ids. That makes re-indexing idempotent — the
same chunk maps to the same point in Qdrant instead of creating duplicates.

In [5]:
# ── [5 / 10] Helper Functions ───────────────────────────────────────────────

def make_chunk_id(document_id: str, chunk_index: int, text: str) -> str:
    """Build a stable, unique id from the document id, chunk index, and text."""
    raw = f"{document_id}:{chunk_index}:{text}"
    return hashlib.sha1(raw.encode("utf-8")).hexdigest()


def build_metadata(chunk: dict, created_at: str) -> dict:
    """Convert a raw chunk into the full enriched record with metadata."""
    return {
        "chunk_id"      : make_chunk_id(
            chunk["document_id"], chunk["chunk_index"], chunk["text"]
        ),
        "document_id"   : chunk["document_id"],
        "chunk_index"   : chunk["chunk_index"],
        "source_file"   : chunk.get("source", ""),
        "document_type" : DOCUMENT_TYPE,
        "title"         : chunk.get("title", ""),
        "created_at"    : created_at,
        "text"          : chunk["text"],
    }


print("Functions defined.")

Functions defined.


---
## 6. Test the Helpers on a Single Chunk

Always test on one example before processing thousands of rows. We enrich the
first chunk, print the result, and confirm the same input always produces the
same `chunk_id`.

In [6]:
# ── [6 / 10] Test the Helpers on a Single Chunk ─────────────────────────────

now_iso = datetime.now(timezone.utc).isoformat()
sample_enriched = build_metadata(chunks[0], now_iso)

print("Enriched chunk:")
print(json.dumps(
    {**sample_enriched, "text": sample_enriched["text"][:80] + "..."},
    indent=2, ensure_ascii=False,
))
print()

# Stable id check — same input must give the same id every time
id_again = make_chunk_id(chunks[0]["document_id"], chunks[0]["chunk_index"], chunks[0]["text"])
assert sample_enriched["chunk_id"] == id_again, "chunk_id is not stable"
print("chunk_id is stable across calls:", sample_enriched["chunk_id"])

Enriched chunk:
{
  "chunk_id": "126011db0a19ac7fc28793640f184fcec5c648c0",
  "document_id": "4983",
  "chunk_index": 0,
  "source_file": "BeIR/scifact",
  "document_type": "scientific_abstract",
  "title": "Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.",
  "created_at": "2026-06-11T16:01:25.151315+00:00",
  "text": "Alterations of the architecture of cerebral white matter in the developing human..."
}

chunk_id is stable across calls: 126011db0a19ac7fc28793640f184fcec5c648c0


---
## 7. Enrich Every Chunk

We now build the metadata for all chunks. We use a single `created_at` timestamp
for the whole run so every chunk from this run shares the same indexing time.

In [7]:
# ── [7 / 10] Enrich Every Chunk ─────────────────────────────────────────────

run_created_at = datetime.now(timezone.utc).isoformat()

enriched = [build_metadata(chunk, run_created_at) for chunk in chunks]

print(f"Enriched {len(enriched):,} chunks")
print(f"created_at for this run : {run_created_at}")

Enriched 12,281 chunks
created_at for this run : 2026-06-11T16:01:35.952335+00:00


---
## 8. Validate the Metadata

Before saving, we confirm two things that the vector store depends on:
1. Every chunk has **all** required fields.
2. Every `chunk_id` is **unique** (no collisions).

In [8]:
# ── [8 / 10] Validate the Metadata ──────────────────────────────────────────

required_keys = {
    "chunk_id", "document_id", "chunk_index", "source_file",
    "document_type", "title", "created_at", "text",
}

missing_fields = sum(1 for c in enriched if not required_keys.issubset(c.keys()))

all_ids    = [c["chunk_id"] for c in enriched]
unique_ids = set(all_ids)
duplicates = len(all_ids) - len(unique_ids)

print(f"Chunks with missing fields : {missing_fields}")
print(f"Total chunk_ids            : {len(all_ids):,}")
print(f"Unique chunk_ids           : {len(unique_ids):,}")
print(f"Duplicate chunk_ids        : {duplicates}")
print()

assert missing_fields == 0, "some chunks are missing required fields"
assert duplicates == 0,     "found duplicate chunk_ids"
print("Metadata validated — all fields present and all ids unique.")

Chunks with missing fields : 0
Total chunk_ids            : 12,281
Unique chunk_ids           : 12,281
Duplicate chunk_ids        : 0

Metadata validated — all fields present and all ids unique.


---
## 9. Save the Enriched Chunks

We write all enriched chunks to `data/processed/04_chunks_with_metadata.jsonl`.
This file is the input to notebook 05, which generates the embeddings.

In [9]:
# ── [9 / 10] Save the Enriched Chunks ───────────────────────────────────────

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with OUTPUT_FILE.open("w", encoding="utf-8") as f:
    for record in enriched:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Saved   : {len(enriched):,} enriched chunks")
print(f"Output  : {OUTPUT_FILE.resolve()}")

Saved   : 12,281 enriched chunks
Output  : /app/data/processed/04_chunks_with_metadata.jsonl


---
## 10. Verify the Output File

We read the file back and confirm the count matches and the metadata fields are
present, then print the first two enriched chunks.

In [10]:
# ── [10 / 10] Verify the Output File ────────────────────────────────────────

loaded = []
with OUTPUT_FILE.open("r", encoding="utf-8") as f:
    for line in f:
        loaded.append(json.loads(line))

print(f"Records in output file : {len(loaded):,}")
assert len(loaded) == len(enriched), "record count mismatch"
assert all(required_keys.issubset(r.keys()) for r in loaded), "missing fields after reload"
print("Output verified — metadata is complete.")
print()

print("=" * 60)
print("First 2 enriched chunks from 04_chunks_with_metadata.jsonl")
print("=" * 60)
for i, record in enumerate(loaded[:2]):
    print(f"\n--- Chunk {i} ---")
    for key, value in record.items():
        display = str(value)[:100] + "..." if len(str(value)) > 100 else value
        print(f"  {key:<14} : {display}")

Records in output file : 12,281
Output verified — metadata is complete.

First 2 enriched chunks from 04_chunks_with_metadata.jsonl

--- Chunk 0 ---
  chunk_id       : 126011db0a19ac7fc28793640f184fcec5c648c0
  document_id    : 4983
  chunk_index    : 0
  source_file    : BeIR/scifact
  document_type  : scientific_abstract
  title          : Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion ten...
  created_at     : 2026-06-11T16:01:35.952335+00:00
  text           : Alterations of the architecture of cerebral white matter in the developing human brain can affect co...

--- Chunk 1 ---
  chunk_id       : 37d98d065f3048e7aa6ccf4be5d01a2fb436db56
  document_id    : 4983
  chunk_index    : 1
  source_file    : BeIR/scifact
  document_type  : scientific_abstract
  title          : Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion ten...
  created_at     : 2026-06-11T16:01:35.952335+00:00
  text   